<a href="https://colab.research.google.com/github/sharniks/langchain-fundamentals/blob/main/langchain_retrievers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN') # Make sure you've added your HF_TOKEN to Colab secrets!

In [9]:
!pip install langchain chromadb faiss-cpu openai tiktoken langchain_huggingface langchain-community wikipedia

# Wikipedia Retriever

In [4]:
from langchain_community.retrievers import WikipediaRetriever

In [5]:
retriever = WikipediaRetriever(top_k_results=2, lang='en')

In [6]:
query = "the geopolitical history of india and pakistan from the perspective of a chinese"

docs = retriever.invoke(query)

In [8]:
for i, doc in enumerate(docs):
  print(f"\n---Result {i+1} ---")
  print(f"Content: \n{doc.page_content}...")


---Result 1 ---
Content: 
The United States has been providing military aid and economic assistance to Pakistan for various purposes since 1948. In 2017, the U.S. stopped military aid to Pakistan, which was about US$2 billion per year. With U.S. military assistance suspended in 2018 and civilian aid reduced to about $300 million for 2022, Pakistani authorities have turned to other countries for help.


== History ==
From 1947 to 1958, under civilian leadership, the United States provided Pakistan with modest economic aid and limited military assistance. During this period, Pakistan became a member of the South East Asian Treaty Organization (SEATO) and the Central Treaty Organization (CENTO), after a Mutual Defence Assistance Agreement signed in May 1954, which facilitated increased levels of both economic and military aid from the U.S.
In 1958, Ayub Khan led Pakistan's first military coup, becoming Chief Martial Law Administrator (CMLA) and later President until 1969. During his tenu

# Vector Store Retriever

In [10]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_core.documents import Document

In [11]:
# Step 1: Your source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [13]:
embedding_model = HuggingFaceEndpointEmbeddings(
    model="BAAI/bge-base-en-v1.5"
)

vectorstore = Chroma.from_documents(
    documents,
    embedding_model,
    collection_name="my_collection"
)

In [14]:
retriever = vectorstore.as_retriever(search_kwargs={"k":2})

In [15]:
query = "What is chroma used for?"
results = retriever.invoke(query)

In [16]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
LangChain helps developers build LLM applications easily.


In [17]:
results = vectorstore.similarity_search(query, k=2)

In [18]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
LangChain helps developers build LLM applications easily.


# MMR

In [19]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [20]:
from langchain_community.vectorstores import FAISS

embedding_model = HuggingFaceEndpointEmbeddings(
    model="BAAI/bge-base-en-v1.5"
)

vectorstore = FAISS.from_documents(
    documents,
    embedding_model
)

In [27]:
from re import search
retriever = vectorstore.as_retriever(
    search_type="mmr",# <-- This enables MMR
    search_kwargs={"k":3, "lambda_mult": 0.5}# k = top results, lambda_mult = relevance-diversity balance
)

In [28]:
query = "What is langchain?"
results = retriever.invoke(query)

In [29]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
LangChain helps developers build LLM applications easily.

--- Result 2 ---
Embeddings convert text into high-dimensional vectors.

--- Result 3 ---
Chroma is a vector database optimized for LLM-based search.


# Multiquery Retriever

In [33]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_core.documents import Document
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

In [34]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [35]:
embedding_model = HuggingFaceEndpointEmbeddings(
    model="BAAI/bge-base-en-v1.5"
)

vectorstore = FAISS.from_documents(documents=all_docs,embedding=embedding_model)

In [36]:
similarity_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [68]:
llm_endpoint = HuggingFaceEndpoint(
    repo_id='openai/gpt-oss-20b',
    task='text-generation'
)
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k":5}),
    llm=ChatHuggingFace(llm=llm_endpoint)
)

In [69]:
# Retrieve results
similarity_results = similarity_retriever.invoke(query)
multiquery_results = multiquery_retriever.invoke(query)

In [70]:
query = "How to improve energy levels and maintain balance?"

In [71]:
# Retrieve results
similarity_results = similarity_retriever.invoke(query)
multiquery_results= multiquery_retriever.invoke(query)

In [72]:
print("\n--- Similarity Results ---")
for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(f"Content: {doc.page_content}")
    print(f"Source: {doc.metadata['source']}")

print("\n--- MultiQuery Results ---")
for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(f"Content: {doc.page_content}")
    print(f"Source: {doc.metadata['source']}")


--- Similarity Results ---

--- Result 1 ---
Content: Drinking sufficient water throughout the day helps maintain metabolism and energy.
Source: H5

--- Result 2 ---
Content: Mindfulness and controlled breathing lower cortisol and improve mental clarity.
Source: H4

--- Result 3 ---
Content: Consuming leafy greens and fruits helps detox the body and improve longevity.
Source: H2

--- Result 4 ---
Content: Regular walking boosts heart health and can reduce symptoms of depression.
Source: H1

--- Result 5 ---
Content: Deep sleep is crucial for cellular repair and emotional regulation.
Source: H3

--- MultiQuery Results ---

--- Result 1 ---
Content: Drinking sufficient water throughout the day helps maintain metabolism and energy.
Source: H5

--- Result 2 ---
Content: Mindfulness and controlled breathing lower cortisol and improve mental clarity.
Source: H4

--- Result 3 ---
Content: Regular walking boosts heart health and can reduce symptoms of depression.
Source: H1

--- Result 4 ---


# ContextualCompressionRetriever

In [73]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEndpointEmbeddings, ChatHuggingFace
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document

In [74]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [75]:
embedding_model = HuggingFaceEndpointEmbeddings(
    model="BAAI/bge-base-en-v1.5"
)

vectorstore = FAISS.from_documents(documents=all_docs,embedding=embedding_model)

In [76]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [86]:
llm_endpoint = HuggingFaceEndpoint(
    repo_id='openai/gpt-oss-20b',
    task='text-generation'
)

# Correctly pass the llm as a keyword argument
chat_model = ChatHuggingFace(llm=llm_endpoint)

compressor = LLMChainExtractor.from_llm(llm=chat_model)

In [87]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

In [88]:
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

In [89]:
for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Photosynthesis enables plants to produce energy by converting sunlight.
